# Contrastive Loss Comparison — Analysis

Анализ результатов сравнения InfoNCE, Triplet и Cosine Similarity loss.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', palette='tab10')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 1. Загрузка результатов

In [ ]:
OUTPUT_DIR = '../outputs'
LOSS_TYPES = ['info_nce', 'triplet', 'cosine']
LOSS_LABELS = {'info_nce': 'InfoNCE', 'triplet': 'Triplet', 'cosine': 'Cosine'}

def load_all_results(output_dir):
    results = []
    for loss_type in LOSS_TYPES:
        loss_dir = os.path.join(output_dir, loss_type)
        if not os.path.isdir(loss_dir):
            continue
        for lr_dir in os.listdir(loss_dir):
            p = os.path.join(loss_dir, lr_dir, 'metrics', 'result.json')
            if os.path.exists(p):
                with open(p) as f:
                    results.append(json.load(f))
    return pd.DataFrame(results)

df = load_all_results(OUTPUT_DIR)
print(f'Loaded {len(df)} runs')
df

## 2. Best Result per Loss

In [ ]:
best = (
    df.sort_values('best_spearman', ascending=False)
    .groupby('loss_type')
    .first()
    .reset_index()
    [['loss_type', 'learning_rate', 'best_spearman', 'training_time', 'epochs_run']]
)
best['Loss'] = best['loss_type'].map(LOSS_LABELS)
best

## 3. Stability Analysis

In [ ]:
stability = (
    df.groupby('loss_type')['best_spearman']
    .agg(['mean', 'std', 'min', 'max'])
    .reset_index()
)
stability.columns = ['Loss Type', 'Mean Spearman', 'Std', 'Min', 'Max']
stability['Loss Type'] = stability['Loss Type'].map(LOSS_LABELS)
stability

## 4. Spearman vs LR

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'info_nce': '#1f77b4', 'triplet': '#ff7f0e', 'cosine': '#2ca02c'}

for loss_type in df['loss_type'].unique():
    sub = df[df['loss_type'] == loss_type].sort_values('learning_rate')
    ax.plot(
        sub['learning_rate'].astype(str),
        sub['best_spearman'],
        marker='o', linewidth=2,
        label=LOSS_LABELS.get(loss_type, loss_type),
        color=colors.get(loss_type)
    )

ax.set_xlabel('Learning Rate')
ax.set_ylabel('Best Spearman')
ax.set_title('Spearman Correlation vs Learning Rate')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Stability Heatmap

In [ ]:
pivot = df.pivot_table(
    index='loss_type', columns='learning_rate', values='best_spearman'
)
pivot.index = [LOSS_LABELS.get(i, i) for i in pivot.index]
pivot.columns = [f'{c:.0e}' for c in pivot.columns]

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax, linewidths=0.5)
ax.set_title('Spearman — Loss × LR')
plt.tight_layout()
plt.show()

## 6. Training Logs

In [ ]:
def load_logs(output_dir, loss_type, lr):
    lr_str = f'lr_{str(lr).replace("-", "neg").replace(".", "_")}'
    base = os.path.join(output_dir, loss_type, lr_str, 'logs')
    train_log = os.path.join(base, 'train_log.csv')
    val_log = os.path.join(base, 'val_log.csv')
    train_df = pd.read_csv(train_log) if os.path.exists(train_log) else None
    val_df = pd.read_csv(val_log) if os.path.exists(val_log) else None
    return train_df, val_df

# Example: plot InfoNCE convergence at different LRs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
loss_type = 'info_nce'
lr_list = [1e-5, 2e-5, 3e-5, 5e-5]

for lr in lr_list:
    train_df, val_df = load_logs(OUTPUT_DIR, loss_type, lr)
    if train_df is not None:
        axes[0].plot(train_df['step'], train_df['train_loss'], label=f'lr={lr:.0e}', linewidth=1.5)
    if val_df is not None:
        axes[1].plot(val_df['epoch'], val_df['spearman_score'], marker='o', label=f'lr={lr:.0e}', linewidth=1.5)

axes[0].set_title(f'{LOSS_LABELS[loss_type]} — Train Loss')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss'); axes[0].legend()
axes[1].set_title(f'{LOSS_LABELS[loss_type]} — Spearman')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Spearman'); axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Embedding Visualization (PCA / t-SNE)

In [ ]:
import torch
from config import load_config
from models.sentence_encoder import SentenceEncoder, get_tokenizer
from evaluation.embedding_metrics import (
    get_embeddings_for_visualization, reduce_with_pca, reduce_with_tsne
)
from training.train_utils import get_device

SAMPLE_SENTENCES = [
    ("A man is playing a guitar.", "music"),
    ("A musician strums an instrument.", "music"),
    ("The guitarist performs on stage.", "music"),
    ("The dog runs across the field.", "animals"),
    ("A cat sleeps on the couch.", "animals"),
    ("Children are playing in the park.", "play"),
    ("Kids enjoy outdoor activities.", "play"),
    ("The weather is sunny today.", "weather"),
    ("Scientists discovered a new planet.", "science"),
    ("Researchers found evidence of life.", "science"),
]

sentences = [s for s, _ in SAMPLE_SENTENCES]
labels = [lbl for _, lbl in SAMPLE_SENTENCES]

# Load InfoNCE best model (adjust path if needed)
cfg = load_config('../experiments/info_nce.yaml')
device = get_device()
tokenizer = get_tokenizer(cfg.model_name)
model = SentenceEncoder(cfg.model_name, cfg.pooling).to(device)

ckpt = '../outputs/info_nce/lr_2_0e-05/checkpoints/best_model.pt'
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()
    
    embeddings = get_embeddings_for_visualization(model, tokenizer, sentences, device)
    pca_2d = reduce_with_pca(embeddings)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    unique_labels = sorted(set(labels))
    palette = sns.color_palette('tab10', n_colors=len(unique_labels))
    color_map = {lbl: palette[i] for i, lbl in enumerate(unique_labels)}
    
    for lbl in unique_labels:
        idx = [i for i, l in enumerate(labels) if l == lbl]
        ax.scatter(pca_2d[idx, 0], pca_2d[idx, 1], label=lbl, color=color_map[lbl], s=80)
    
    for i, sent in enumerate(sentences):
        ax.annotate(sent[:20] + '...', (pca_2d[i, 0], pca_2d[i, 1]), fontsize=7, alpha=0.7)
    
    ax.set_title('PCA of Sentence Embeddings (InfoNCE)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f'Checkpoint not found: {ckpt}\nRun training first.')

## 8. Выводы исследования

После запуска экспериментов заполните эту секцию:

### Лучшая функция потерь
- По Spearman корреляции: **[заполнить]**
- По скорости сходимости: **[заполнить]**  
- По стабильности к LR: **[заполнить]**

### Влияние Learning Rate
- InfoNCE: **[заполнить]**
- Triplet: **[заполнить]**
- Cosine: **[заполнить]**

### Trade-off: качество vs скорость
- **[заполнить]**